In [ ]:
import pandas as pd

In [ ]:
# %%bash 

# wget "http://hgdownload.cse.ucsc.edu/goldenPath/hg38/database/refGene.txt.gz"

In [ ]:
coords = pd.read_table('refGene.txt.gz', header = None)

In [ ]:
gene_df = coords[[2, 4, 5, 12]]

In [ ]:
gene_df.columns = ['chrom', 'start', 'end', 'gene']

In [ ]:
mca_pos = pd.read_csv('mca_position_0116.csv')

In [ ]:
mca_pos['q_arm_bool'] = False
mca_pos.loc[(mca_pos['q_arm'] == 'T'),'q_arm_bool'] = True
mca_pos.loc[(mca_pos['q_arm'] == 'Y'),'q_arm_bool'] = True
mca_pos.loc[(mca_pos['q_arm'] == 'TRUE'),'q_arm_bool'] = True
mca_pos['p_arm_bool'] = False
mca_pos.loc[(mca_pos['p_arm'] == 'T'),'p_arm_bool'] = True
mca_pos.loc[(mca_pos['p_arm'] == 'Y'),'p_arm_bool'] = True
mca_pos.loc[(mca_pos['p_arm'] == 'TRUE'),'p_arm_bool'] = True
mca_pos.loc[(mca_pos['p_arm'] == 'C'),'p_arm_bool'] = True
mca_pos['botharms_bool'] = False
mca_pos.loc[(mca_pos['p_arm_bool']==True) & (mca_pos['q_arm_bool']==True), 'botharms_bool'] = True
mca_pos['arm_short'] = ''
mca_pos.loc[((mca_pos['p_arm_bool']) == True & (mca_pos['q_arm_bool']==False)),'arm_short'] = 'p'
mca_pos.loc[((mca_pos['p_arm_bool']) == False & (mca_pos['q_arm_bool']==True)),'arm_short'] = 'q'
mca_pos.loc[(mca_pos['botharms_bool'] == True),'arm_short'] = ''

In [ ]:
mca_pos = mca_pos[(mca_pos['cohort']!='BioVU') &
                  (mca_pos['type']!='Undetermined')]

In [ ]:
mca_pos[['sample_id', 'chrom', 'type', 'arm_short', 'beg_GRCh38', 'end_GRCh38', 'cohort']]

In [ ]:
import pandas as pd
from collections import defaultdict

def find_overlapping_genes(gene_df, chrom, start, end):
    """
    Find genes that overlap with the given genomic region.
    """
    overlapping = gene_df[
        (gene_df['chrom'] == chrom) & 
        (gene_df['start'] <= end) & 
        (gene_df['end'] >= start)
    ]
    return set(overlapping['gene'].tolist())

def find_shared_regions(data, threshold):
    """
    Find genomic regions that are shared among at least threshold% of mCAs.
    A unique mCA is defined by chromosome + arm (if any) + type.
    """
    # Create a unique identifier for each mCA configuration
    data['mca_config'] = data.apply(
        lambda x: f"{x['chrom']}{x['arm_short']}_{x['type']}", axis=1
    )
    
    # Group by mCA configuration
    grouped = data.groupby('mca_config')
    shared_regions = defaultdict(list)
    
    for mca_config, group in grouped:
        if len(group) < 2:  # Skip if only one mCA of this configuration
            continue
            
        # Get all unique positions
        positions = []
        for _, row in group.iterrows():
            positions.extend([
                (row['beg_GRCh38'], 'start'), 
                (row['end_GRCh38'], 'end')
            ])
        positions.sort()
        
        # Count overlaps
        current_count = 0
        last_pos = None
        min_required = threshold * len(group)
        
        # Extract chromosome and arm
        chrom = group.iloc[0]['chrom']
        arm = group.iloc[0]['arm_short']
        mca_type = group.iloc[0]['type']
        
        for pos, event_type in positions:
            if event_type == 'start':
                current_count += 1
                if current_count >= min_required and last_pos is not None:
                    shared_regions[mca_config].append({
                        'chrom': chrom,
                        'arm': arm,
                        'type': mca_type,
                        'start': last_pos,
                        'end': pos,
                        'coverage': current_count / len(group),
                        'total_events': len(group)
                    })
            else:
                if current_count >= min_required:
                    shared_regions[mca_config].append({
                        'chrom': chrom,
                        'arm': arm,
                        'type': mca_type,
                        'start': last_pos,
                        'end': pos,
                        'coverage': current_count / len(group),
                        'total_events': len(group)
                    })
                current_count -= 1
            last_pos = pos
            
    return shared_regions

def merge_adjacent_regions(regions, max_gap=1000):
    """Merge adjacent or nearly adjacent regions."""
    if not regions:
        return []
        
    sorted_regions = sorted(regions, key=lambda x: (x['chrom'], x['start']))
    merged = [sorted_regions[0]]
    
    for current in sorted_regions[1:]:
        previous = merged[-1]
        
        if (previous['chrom'] == current['chrom'] and 
            current['start'] - previous['end'] <= max_gap and
            previous['arm'] == current['arm']):
            previous['end'] = max(previous['end'], current['end'])
            previous['coverage'] = min(previous['coverage'], current['coverage'])
        else:
            merged.append(current)
            
    return merged

def analyze_shared_regions(mca_data, gene_df, thresholds=[0.5, 0.75, 0.8, 0.9]):
    """
    Analyze shared regions at different overlap thresholds and find overlapping genes.
    """
    results = {}
    
    for threshold in thresholds:
        shared = find_shared_regions(mca_data, threshold)
        
        merged_results = {}
        for mca_config, regions in shared.items():
            merged = merge_adjacent_regions(regions)
            
            # Add overlapping genes to each region
            for region in merged:
                region['genes'] = find_overlapping_genes(
                    gene_df,
                    region['chrom'], 
                    region['start'], 
                    region['end']
                )
            
            merged_results[mca_config] = merged
            
        results[threshold] = merged_results
        
    return results

def format_results(results):
    """Format results into a readable string with gene information."""
    output = []
    for threshold, mca_regions in sorted(results.items()):
        output.append(f"\nShared regions at {threshold*100}% threshold:")
        
        for mca_config, regions in sorted(mca_regions.items()):
            # Parse mca_config back into readable format
            arm_desc = " whole chromosome" if regions[0]['arm'] == "" else f" {regions[0]['arm']} arm"
            output.append(f"\n{regions[0]['chrom']}{arm_desc} - {regions[0]['type']}:")
            
            for region in regions:
                output.append(
                    f"  Region {region['start']:,}-{region['end']:,} "
                    f"(Coverage: {region['coverage']*100:.1f}%, "
                    f"Events: {region['total_events']})"
                )
                
                if region['genes']:
                    output.append("    Genes in region:")
                    genes = sorted(region['genes'])  # Sort alphabetically
                    for i in range(0, len(genes), 4):  # Show 4 genes per line
                        gene_line = genes[i:i+4]
                        output.append(f"      {', '.join(gene_line)}")
                else:
                    output.append("    No genes found in this region")
                output.append("")  # Add blank line between regions
                
    return '\n'.join(output)

def run_analysis(mca_pos, gene_df):
    """Run the complete analysis using mCA positions and gene data."""
    results = analyze_shared_regions(mca_pos, gene_df)
    return format_results(results)

In [ ]:
results = analyze_shared_regions(mca_pos, gene_df)

In [ ]:
# x = pd.read_table("frequently-mutated-genes.2025-01-19.tsv")
# x[(x['num_cohort_ssm_affected_cases']>5) &
#   (x['num_gdc_ssm_affected_cases']>5) &
#   (x['num_cohort_cnv_gain_cases']>5) & (x['num_cohort_cnv_loss_cases']>5)]

In [ ]:
# blood_cancer_genes_nci = set(pd.read_table("frequently-mutated-genes.2025-01-19.tsv")['symbol'].to_list())

In [ ]:
cbio_gene_df = pd.read_table('Mutated_Genes.txt')
cbio_genes = set(cbio_gene_df[cbio_gene_df['#']>5].Gene.to_list())

cbio_cna_df = pd.read_table('CNA_Genes.txt')
blood_cancer_genes = cbio_genes.union(set(cbio_cna_df[cbio_cna_df['Profiled Samples']>5].Gene.to_list()))

In [ ]:
len(blood_cancer_genes)

In [ ]:
len(blood_cancer_genes_nci)

In [ ]:
mca_cancer_genes_dict = dict()
mca_thresholds = dict()
thresholds = [0.5, 0.75, 0.8, 0.9]

for key, value in results[thresholds[len(thresholds)-1]].items():
    i=len(thresholds)-1
    genes = value[0]['genes']
    cancer_genes_mca = genes.intersection(blood_cancer_genes)
    while len(cancer_genes_mca) == 0 and i>=0:
        print(key)
        print(i)
        i = i - 1
        genes = results[thresholds[i]][key][0]['genes']
        cancer_genes_mca = genes.intersection(blood_cancer_genes)
        print(cancer_genes_mca)
    threshold = thresholds[i]
    mca_cancer_genes_dict[key] = cancer_genes_mca
    mca_thresholds[key] = threshold
    
rows = []

for key, value in mca_cancer_genes_dict.items():
    row = {
        'mca': key,
        'threshold': mca_thresholds[key],
        'cbio_cancer_genes': '; '.join(sorted(list(value))) if len(value) > 0 else 'No genes'
    }
    rows.append(row)
    
df = pd.DataFrame(rows)

In [ ]:
df.to_csv('mcas_mapped_to_cbio_cancer_genes.tsv', sep='\t')

In [ ]:
cancer_gene_labels = pd.read_table('cancerGeneList.tsv')

In [ ]:
cancer_gene_labels = cancer_gene_labels[['Hugo Symbol', 'Is Oncogene', 'Is Tumor Suppressor Gene']]

In [ ]:
cancer_gene_label_dict = dict()

for index, row in cancer_gene_labels.iterrows():
    gene = row['Hugo Symbol']
    label = "none"
    if row['Is Oncogene'] == 'Yes' and row['Is Tumor Suppressor Gene'] == 'No':
        label = "oncogene"
    if row['Is Oncogene'] == 'No' and row['Is Tumor Suppressor Gene'] == 'Yes':
        label = "tumor suppressor"
    if row['Is Oncogene'] == 'Yes' and row['Is Tumor Suppressor Gene'] == 'Yes':
        label = "both"
        
    cancer_gene_label_dict[gene] = label

In [ ]:
mca_cancer_genes_dict_labeled = dict()

for mca, value in mca_cancer_genes_dict.items():
    gene_label_set = set()
    for gene in value:
        gene_label = gene + "_" + cancer_gene_label_dict[gene]
        gene_label_set.add(gene_label)
    mca_cancer_genes_dict_labeled[mca]=gene_label_set

In [ ]:
mca_cancer_genes_dict_labeled

In [ ]:
rows = []

for key, value in mca_cancer_genes_dict_labeled.items():
    row = {
        'mca': key,
        'cbio_cancer_genes_labeled': '; '.join(sorted(list(value))) if len(value) > 0 else 'No genes'
    }
    rows.append(row)
    
df = pd.DataFrame(rows)

In [ ]:
df.to_csv('mca_shared_genes_labeled_n>5.tsv', sep='\t')

In [ ]:
import pandas as pd
from collections import defaultdict

# [Previous code for find_overlapping_genes, find_shared_regions, merge_adjacent_regions remains the same]

def results_to_dataframe(results, threshold=0.90):
    """
    Convert results dictionary at specified threshold to a DataFrame.
    
    Parameters:
    results (dict): Results dictionary from analyze_shared_regions
    threshold (float): Threshold to extract (default 0.90)
    
    Returns:
    pd.DataFrame: DataFrame with columns for region information and genes
    """
    threshold_data = results[threshold]
    rows = []
    
    for mca_config, regions in threshold_data.items():
        for region in regions:
            # Create row dictionary
            row = {
                'chromosome': region['chrom'],
                'arm': 'whole' if region['arm'] == '' else region['arm'],
                'type': region['type'],
                'start': region['start'],
                'end': region['end'],
                'coverage': region['coverage'],
                'total_events': region['total_events'],
                'genes': '; '.join(sorted(region['genes'])) if region['genes'] else 'No genes'
            }
            rows.append(row)
    
    # Create DataFrame and sort by chromosome and start position
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['chromosome', 'start'])
        
        # Convert coverage to percentage
        df['coverage'] = df['coverage'] * 100
        
        # Format columns
        df = df.rename(columns={
            'chromosome': 'chrom',
            'coverage': 'coverage_percent'
        })
        
        # Reorder columns
        column_order = [
            'chrom', 'arm', 'type', 'start', 'end', 
            'coverage_percent', 'total_events', 'genes'
        ]
        df = df[column_order]
    
    return df

def analyze_shared_regions(mca_data, gene_df, thresholds=[0.5, 0.75, 0.8, 0.9, 0.99]):
    """
    Analyze shared regions and return both dictionary and DataFrame results.
    
    Returns:
    tuple: (results_dict, results_df_90) 
    """
    results = {}
    
    for threshold in thresholds:
        shared = find_shared_regions(mca_data, threshold)
        
        merged_results = {}
        for mca_config, regions in shared.items():
            merged = merge_adjacent_regions(regions)
            
            # Add overlapping genes to each region
            for region in merged:
                region['genes'] = find_overlapping_genes(
                    gene_df,
                    region['chrom'], 
                    region['start'], 
                    region['end']
                )
            
            merged_results[mca_config] = merged
            
        results[threshold] = merged_results
    
    # Create DataFrame for 90% threshold
    results_df_90 = results_to_dataframe(results, threshold=0.90)
    
    return results, results_df_90

def run_analysis(mca_pos, gene_df):
    """
    Run the complete analysis and return both text results and DataFrame.
    
    Returns:
    tuple: (results_text, results_df_90)
    """
    results_dict, results_df_90 = analyze_shared_regions(mca_pos, gene_df)
    results_text = format_results(results_dict)
    return results_text, results_df_90

# Example usage:
results_text, results_df_90 = run_analysis(mca_pos, gene_df)
# print("\nDataFrame of 90% shared regions:")
# print(results_df_90)

In [ ]:
results_df_90.to_csv('mca_shared_genes.tsv', sep='\t')

In [ ]:
threshold_df = pd.read_table('mcas_mapped_to_cbio_cancer_genes.tsv')[['mca', 'threshold']]

In [ ]:
threshold_df

In [ ]:
mca_msar_df = pd.read_table('mca_shared_genes.tsv')[['chrom', 'arm', 'type', 'start', 'end', 'total_events']]

In [ ]:
# Function to create 'mca' column
def create_mca(row):
    if row['arm'] == 'whole':
        return f"{row['chrom']}_{row['type']}"
    else:
        return f"{row['chrom']}{row['arm']}_{row['type']}"

# Apply the function to each row
mca_msar_df['mca'] = mca_msar_df.apply(create_mca, axis=1)

In [ ]:
x = mca_msar_df.merge(threshold_df, how='inner', on='mca')[['mca', 'threshold', 'start', 'end', 'total_events']]

x.columns = ['mca', 'threshold', 'start_pos_hg38', 'end_pos_hg38', 'total_events']

In [ ]:
x.to_csv('mca_1m_supptable2.csv')